# Milestone 1

This milestone focuses on understanding the dataset and establishing a baseline performance through **exploratory data analysis (EDA)** and simple **heuristic-based methods** using `librosa`.

---

## Suggested Readings
- [Hugging Face Audio Course](https://huggingface.co/learn/audio-course/en/chapter0/introduction)
- [Librosa Documentation](https://librosa.org/doc/main/core.html#audio-loading)

---

## Instructions
Use this notebook to answer **all Milestone-1 questions**.

---

## Resources
- Notebook Link:  
  https://colab.research.google.com/drive/1m6UczhxQIke_raWSqukSWuiKbIVt7MMb?usp=sharing  

- Competition Link:  
  https://www.kaggle.com/competitions/jan-2026-dl-gen-ai-project/


In [64]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
import librosa.display
import matplotlib.pyplot as plt
import random
import torch

import warnings
warnings.filterwarnings("ignore")

In [65]:
#----------------------------- DON'T CHANGE THIS --------------------------
DATA_SEED = 67
TRAINING_SEED = 1234
SR = 22050
DURATION = 5.0
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
TOP_DB=20
TARGET_SNR_DB = 10

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)
torch.manual_seed(DATA_SEED)
torch.cuda.manual_seed(DATA_SEED)

In [66]:
# CONFIGURATION
DATA_ROOT = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock'] # Make the list of all genres available (alphabetical order)
STEMS = {'drums': 'drums.wav', 'vocals': 'vocals.wav', 'bass': 'bass.wav', 'other': 'other.wav'} # Write here stems file name
STEM_KEYS = ['drums', 'vocals', 'bass', 'other']
GENRE_TO_TEST = 'rock'
# SONG_INDEX = #Enter index as per Q10.

In [67]:
# Question - 1
# What is the value of total number of corrupted sounds ( less than 4kb) + (total number of sounds < 5.0491MB)

import os

CORRUPTED_LIMIT = 4096
SIZE_LIMIT = 5294365

corrupted_count = 0
small_count = 0

for root, dirs, files in os.walk(DATA_ROOT+'/genres_stems'):
    for file in files:
        if file.lower().endswith(".wav"):
            file_path = os.path.join(root, file)
            try:
                size = os.path.getsize(file_path)

                if size < CORRUPTED_LIMIT:
                    corrupted_count += 1

                if size < SIZE_LIMIT:
                    small_count += 1

            except OSError:
                continue

total = corrupted_count + small_count

print(f"Corrupted wav files (<4KB): {corrupted_count}")
print(f"Wav files <5.0491MB: {small_count}")
print(f"Total: {total}")


Corrupted wav files (<4KB): 0
Wav files <5.0491MB: 1256
Total: 1256


In [68]:
# Question 2
# What is the absolute difference between  total number of sounds > 5.0493MB and total number of sounds < 5.0491MB ?

import os

UPPER_LIMIT = 5294575   
LOWER_LIMIT = 5294365  

greater_count = 0
smaller_count = 0

for root, dirs, files in os.walk(DATA_ROOT+'/genres_stems'):
    for file in files:
        if file.lower().endswith(".wav"):
            file_path = os.path.join(root, file)
            try:
                size = os.path.getsize(file_path)

                if size > UPPER_LIMIT:
                    greater_count += 1

                if size < LOWER_LIMIT:
                    smaller_count += 1

            except OSError:
                continue

abs_diff = abs(greater_count - smaller_count)

print(f"Wav files >5.0493MB: {greater_count}")
print(f"Wav files <5.0491MB: {smaller_count}")
print(f"Absolute Difference: {abs_diff}")


Wav files >5.0493MB: 184
Wav files <5.0491MB: 1256
Absolute Difference: 1072


In [78]:
import os
import random
from pathlib import Path

# --- CONFIGURATION ---
DATA_ROOT = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']
STEMS = {'drums': 'drums.wav', 'vocals': 'vocals.wav', 'bass': 'bass.wav', 'other': 'other.wav'}
STEM_KEYS = ['drums', 'vocals', 'bass', 'other']
GENRE_TO_TEST = 'rock'

def build_dataset(root_dir, val_split=0.17, seed=42):
    # Initialize the nested dictionary structure
    train_dataset = {g: {k: [] for k in STEM_KEYS} for g in GENRES}
    val_dataset   = {g: {k: [] for k in STEM_KEYS} for g in GENRES}

    rng = random.Random(seed)
    CORRUPT_LIMIT = 4096 # 4KB
    
    # Path to the specific subfolder shown in your image
    base_path = os.path.join(root_dir, 'genres_stems')

    for genre in GENRES:
        genre_path = os.path.join(base_path, genre)

        if not os.path.isdir(genre_path):
            continue

        valid_songs = []

        # 1. Gather all valid song folders (e.g., blues.00000)
        for song_folder in sorted(os.listdir(genre_path)):
            song_path = os.path.join(genre_path, song_folder)

            if not os.path.isdir(song_path):
                continue

            # Check if all required stems exist and are not corrupted
            song_stems = {}
            is_valid = True

            for key, filename in STEMS.items():
                stem_path = os.path.join(song_path, filename)
                
                if not os.path.isfile(stem_path) or os.path.getsize(stem_path) < CORRUPT_LIMIT:
                    is_valid = False
                    break
                
                song_stems[key] = stem_path

            if is_valid:
                valid_songs.append(song_stems)

        # 2. Shuffle the full song groups (keeping stems together)
        rng.shuffle(valid_songs)

        # 3. Calculate Split
        split_idx = int(len(valid_songs) * (1 - val_split))
        train_songs = valid_songs[:split_idx]
        val_songs = valid_songs[split_idx:]
        
        # This confirms why you see '6' (it prints per genre)
        print(f"Genre: {genre:10} | Train: {len(train_songs):3} | Val: {len(val_songs):3}")

        # 4. Helper to populate the nested dictionary structure
        def add_to_dict(target_dict, song_list):
            for song_entry in song_list:
                for key in STEM_KEYS:
                    # Append the path to the correct genre and stem key
                    target_dict[genre][key].append(song_entry[key])

        add_to_dict(train_dataset, train_songs)
        add_to_dict(val_dataset, val_songs)

    return train_dataset, val_dataset

# Execute
tr, val = build_dataset(DATA_ROOT)
tr['blues']['drums'][1]

Genre: blues      | Train:  83 | Val:  17
Genre: classical  | Train:  83 | Val:  17
Genre: country    | Train:  83 | Val:  17
Genre: disco      | Train:  83 | Val:  17
Genre: hiphop     | Train:  83 | Val:  17
Genre: jazz       | Train:  83 | Val:  17
Genre: metal      | Train:  83 | Val:  17
Genre: pop        | Train:  83 | Val:  17
Genre: reggae     | Train:  83 | Val:  17
Genre: rock       | Train:  83 | Val:  17


'/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/blues/blues.00041/drums.wav'

In [70]:
# Question 3 
# What is the absolute difference between the number of training reggae drum samples and the number of validation country vocal samples?

train_reggae_drum = len(tr["reggae"]["drums"])
val_country_vocal = len(val["country"]["vocals"])

abs_difference = abs(train_reggae_drum - val_country_vocal)

print("Training Reggae Drum Samples:", train_reggae_drum)
print("Validation Country Vocal Samples:", val_country_vocal)
print("Absolute Difference:", abs_difference)


Training Reggae Drum Samples: 83
Validation Country Vocal Samples: 17
Absolute Difference: 66


In [71]:
import librosa
import pandas as pd
import numpy as np
import os

def find_long_silences(dataset_dict, sr=SR, threshold_sec=DURATION, top_db=TOP_DB):
    records = []
    
    all_tasks = []
    for genre, stems in dataset_dict.items():
        for stem_name, paths in stems.items():
            for path in paths:
                all_tasks.append((genre, stem_name, path))
    
    total_files = len(all_tasks)
    print(f"Starting analysis...")

    for idx, (genre, stem_name, file_path) in enumerate(all_tasks):
        y, _ = librosa.load(file_path, sr=sr)
        total_duration = librosa.get_duration(y=y, sr=sr)
        
        intervals = librosa.effects.split(y, top_db=top_db)
        
        max_silence = 0
        silence_types = []

        if len(intervals) == 0:
            max_silence = total_duration
            silence_types.append("Full")
        else:
            intervals_sec = intervals / sr
            
            start_silence = intervals_sec[0][0]
            if start_silence >= threshold_sec:
                silence_types.append("Start")
                max_silence = max(max_silence, start_silence)

            for i in range(len(intervals_sec) - 1):
                mid_silence = intervals_sec[i+1][0] - intervals_sec[i][1]
                if mid_silence >= threshold_sec:
                    silence_types.append("Middle")
                    max_silence = max(max_silence, mid_silence)

            end_silence = total_duration - intervals_sec[-1][1]
            if end_silence >= threshold_sec:
                silence_types.append("End")
                max_silence = max(max_silence, end_silence)

        if max_silence >= threshold_sec:
            records.append({
                "Genre": genre,
                "Stem": stem_name,
                "Duration": round(total_duration, 2),
                "Max_Silence_Sec": round(max_silence, 2),
                "Silence_Location": ", ".join(list(set(silence_types))),
                "File_Path": file_path
            })
            
        if (idx + 1) % 50 == 0:
            print(f"Processed {idx + 1}/{total_files} files...")

    return pd.DataFrame(records)

df_silence = find_long_silences(tr, sr=SR, threshold_sec=DURATION, top_db=TOP_DB)


print("\n" + "="*60)
print("EXTENDED DATASET QUALITY REPORT")
print("="*60)

if not df_silence.empty:
    total_silent_files = len(df_silence)
    
    total_vocal_silence = len(df_silence[df_silence['Stem'] == 'vocals'])
    
    avg_vocal_silence = df_silence[df_silence['Stem'] == 'vocals']['Max_Silence_Sec'].mean()
    
    jazz_drums_df = df_silence[(df_silence['Genre'] == 'jazz') & (df_silence['Stem'] == 'drums')]
    jazz_drums_silence = len(jazz_drums_df)

    jazz_drums_middle_only = len(jazz_drums_df[jazz_drums_df['Silence_Location'] == 'Middle'])

    jazz_drums_10s = len(jazz_drums_df[jazz_drums_df['Max_Silence_Sec'] >= 10])

    print(f"Total files with silence >= 5s:              {total_silent_files}")
    print(f"Total 'Vocals' with silence >= 5s:           {total_vocal_silence}")
    print(f"Average silence length in 'Vocals':          {avg_vocal_silence:.2f}s")
    print(f"Total 'Drums' in 'Jazz' with silence >= 5s:  {jazz_drums_silence}")
    print(f"Total 'Jazz Drums' (ONLY Middle silence):    {jazz_drums_middle_only}")
    print(f"Total 'Jazz Drums' (Max Silence >= 10s):     {jazz_drums_10s}")
    
    print("\n Pivot Table (Genre vs Stem)")
    pivot = df_silence.pivot_table(index='Genre', columns='Stem', values='File_Path', aggfunc='count', fill_value=0)
    print(pivot)

else:
    print("No match detected")
    

Starting analysis of 3320 files...
Processed 50/3320 files...
Processed 100/3320 files...
Processed 150/3320 files...
Processed 200/3320 files...
Processed 250/3320 files...
Processed 300/3320 files...
Processed 350/3320 files...
Processed 400/3320 files...
Processed 450/3320 files...
Processed 500/3320 files...
Processed 550/3320 files...
Processed 600/3320 files...
Processed 650/3320 files...
Processed 700/3320 files...
Processed 750/3320 files...
Processed 800/3320 files...
Processed 850/3320 files...
Processed 900/3320 files...
Processed 950/3320 files...
Processed 1000/3320 files...
Processed 1050/3320 files...
Processed 1100/3320 files...
Processed 1150/3320 files...
Processed 1200/3320 files...
Processed 1250/3320 files...
Processed 1300/3320 files...
Processed 1350/3320 files...
Processed 1400/3320 files...
Processed 1450/3320 files...
Processed 1500/3320 files...
Processed 1550/3320 files...
Processed 1600/3320 files...
Processed 1650/3320 files...
Processed 1700/3320 files...

In [96]:
stems_audio = []
SONG_INDEX=0

try:
    for key in STEM_KEYS:
        file_path = tr[GENRE_TO_TEST][key][SONG_INDEX]
        y, _ = librosa.load(file_path, sr=SR, duration=DURATION)
        stems_audio.append(y)

    print("Audio loaded successfully.")
except NameError:
    print("ERROR: 'tr' dictionary not found. Please run build_dataset() first.")
except IndexError:
    print(f"ERROR: Song index {SONG_INDEX} out of range for genre {GENRE_TO_TEST}.")
except Exception as e:
    print(f"ERROR: {e}")

Audio loaded successfully.


In [97]:
# ------------------- write your code here -------------------------------
# Stack them into a numpy array (Shape: 4 x Samples)
stems_stack = np.array(stems_audio)

# Mix the stems by summing them element-wise
mix_raw = np.sum(stems_stack, axis=0)

# Calculate RMS Amplitude MANUALLY
rms_val = np.sqrt(np.mean(mix_raw**2))

#Peak Normalization
max_val = np.max(np.abs(mix_raw))

if max_val > 0:
    mix_norm = mix_raw / max_val
else:
    mix_norm = mix_raw

# VALIDATION
assert np.isclose(np.max(np.abs(mix_norm)), 1.0), "Normalization failed."
#------------------------------------------------------------------------


In [98]:
# Output (10-12)

print(f"Length of mix sample:        {len(mix_norm)} samples")
print(f"RMS Amplitude of mix:        {rms_val:.4f}")
print(f"Max value (Peak Normalized): {np.max(np.abs(mix_norm))}")

RESULTS FOR QUESTIONS 10-12
10. Length of mix sample:        110250 samples
11. RMS Amplitude of mix:        0.1021
12. Max value (Peak Normalized): 1.0
